# Installing EdgeR & limma

In [3]:
if (!require("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager", repos = "http://cran.us.r-project.org")
}

BiocManager::install(c("edgeR", "limma"))

Bioconductor version '3.18' is out-of-date; the current release version '3.23'
  is available with R version '4.6'; see https://bioconductor.org/install

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.18 (BiocManager 1.30.26), R 4.3.3 (2024-02-29)

Installing package(s) 'edgeR', 'limma'

installiere auch Abhängigkeiten ‘locfit’, ‘statmod’


Aktualisiere HTML-Index der Pakete in '.Library'

Making 'packages.html' ...
 fertig

Old packages: 'abind', 'base64enc', 'BH', 'BiocManager', 'bslib', 'callr',
  'cli', 'clipr', 'cluster', 'colorspace', 'cpp11', 'curl', 'data.table',
  'devtools', 'digest', 'downlit', 'dqrng', 'ellipsis', 'fansi', 'fastDummies',
  'fitdistrplus', 'fs', 'future', 'future.apply', 'gert', 'ggplot2', 'ggrepel',
  'gh', 'globals', 'glue', 'gplots', 'gridExtra', 'here', 'highr', 'htmltools',
  'httpuv',

# EdgeR: Each Condition vs 3month

In [1]:
# EDGER PIPELINE
options(warn = -1)
options(stringsAsFactors = FALSE)
library(edgeR)
library(limma)
library(stringr)
library(dplyr)

# 1. CONFIGURATION
base_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month"
pb_dir   <- file.path(base_dir, "pseudobulks")

out_raw  <- file.path(base_dir, "results/edgeR_raw")
out_sig  <- file.path(base_dir, "results/edgeR_significant")

dir.create(out_raw, recursive = TRUE, showWarnings = FALSE)
dir.create(out_sig, recursive = TRUE, showWarnings = FALSE)

ref_condition <- "3month-protocol"
conditions_to_test <- c("6month-protocol", "Prematurely-terminated", "PCMV/PRV", "Rejection")

# Load clinical info
meta_full <- read.csv(file.path(base_dir, "meta_info.csv"), stringsAsFactors = FALSE)
rownames(meta_full) <- meta_full$Sample_ID
# Rename to match the exact supergrouping column logic (Supergroupping: Disease contains all xenografts and Control has all the non-transplanted hearts)
colnames(meta_full)[which(colnames(meta_full) == "Condition")] <- "Disease"

# 2. Main loop across all cell types
celltype_files <- list.files(pb_dir, pattern = "\\.csv$")

for (ct_file in celltype_files) {
  celltype <- gsub("\\.csv", "", ct_file)
  cat("\n=========================================================\n")
  cat("PROCESSING CELL TYPE:", celltype, "\n")
  cat("=========================================================\n")
  
  # Load pseudobulk matrix
  pesudo_count <- read.csv(file.path(pb_dir, ct_file), check.names = FALSE, row.names = 1)
  
  # Loop over all comparison groups against Ref_condition
  for (cond in conditions_to_test) {
    cat("  -> Comparing:", cond, "vs", ref_condition, "\n")
    
    # 1. Subset metadata for this specific comparison
    sub_meta <- meta_full[meta_full$Disease %in% c(ref_condition, cond), , drop = FALSE]
    
    # 2. Select valid samples (colSums > 0 AND present in sub_meta)
    valid_samples <- intersect(colnames(pesudo_count)[colSums(pesudo_count) > 0], rownames(sub_meta))
    
    if (length(valid_samples) == 0) {
      cat("     SKIP: No valid samples.\n")
      next
    }
    
    mat <- pesudo_count[, valid_samples, drop = FALSE]
    meta <- sub_meta[valid_samples, , drop = FALSE]
    
    # 3. Filter genes (sum > 0 across selected samples)
    mat <- mat[rowSums(mat) > 0, , drop = FALSE]
    
    # 4. Check Group Sizes (Must be > 2 per colleague's exact logic)
    n_ref  <- sum(meta$Disease == ref_condition)
    n_cond <- sum(meta$Disease == cond)
    
    if (n_ref > 2 && n_cond > 2) {
      
      
      # EDGER ANALYSIS CORE
      
      deglist <- DGEList(counts = mat, group = as.factor(meta$Disease))
      
      # 5. Filter low expression genes (mean > 0.0125 in AT LEAST ONE group)
      idx_ref  <- deglist$samples$group == ref_condition
      idx_cond <- deglist$samples$group == cond
      
      mean_ref  <- rowMeans(deglist$counts[, idx_ref, drop = FALSE])
      mean_cond <- rowMeans(deglist$counts[, idx_cond, drop = FALSE])
      
      keep <- which(mean_ref > 0.0125 | mean_cond > 0.0125)
      deglist_keep <- deglist[keep, keep.lib.sizes = FALSE]
      
      # 6. Normalize and model
      deglist_norm <- calcNormFactors(deglist_keep, method = "TMM")
      
      # Use R's make.names to handle special characters (like / or - in conditions)
      safe_disease <- make.names(meta$Disease)
      design <- model.matrix(~0 + safe_disease)
      colnames(design) <- gsub("safe_disease", "", colnames(design))
      
      deg <- estimateDisp(deglist_norm, design, robust = TRUE)
      fit <- glmQLFit(deg, design)
      
      # Build contrast dynamically using the safe names
      safe_cond <- make.names(cond)
      safe_ref  <- make.names(ref_condition)
      contrast_str <- paste(safe_cond, safe_ref, sep = "-")
      
      compare <- makeContrasts(contrasts = contrast_str, levels = design)
      qlf <- glmQLFTest(fit, contrast = compare)
      
      # 7. Extract Results
      res <- topTags(qlf, n = nrow(deglist_norm$counts))$table
      
      # Add observation counts
      res[, paste0("Observations_", cond)] <- n_cond
      res[, paste0("Observations_", ref_condition)] <- n_ref
      
      # Add metadata for easy concatenation later
      res$Gene <- rownames(res)
      res$CellType <- celltype
      res$Comparison <- paste(cond, "vs", ref_condition)
      
      # Reorder columns to make it clean
      res <- res[, c("Gene", "CellType", "Comparison", "logFC", "logCPM", "F", "PValue", "FDR", 
                     paste0("Observations_", cond), paste0("Observations_", ref_condition))]
      
      
      # ORGANIZE DIRECTORIES AND EXPORT
      
      # Create a safe folder name for this specific comparison
      comp_folder_name <- paste0(gsub("[^A-Za-z0-9_]", "_", cond), "_vs_", ref_condition)
      
      # Export Raw Results
      comp_raw_dir <- file.path(out_raw, comp_folder_name)
      dir.create(comp_raw_dir, recursive = TRUE, showWarnings = FALSE)
      
      raw_filename <- paste0("edgeR_Raw_", celltype, "_", comp_folder_name, ".csv")
      write.csv(res, file.path(comp_raw_dir, raw_filename), row.names = FALSE)
      
      
      # FILTER, SORT & EXPORT SIGNIFICANT RESULTS
      
      # Apply FDR < 0.05 and |logFC| >= 0.5
      res_sig <- res[res$FDR < 0.05 & abs(res$logFC) >= 0.5, ]
      
      if (nrow(res_sig) > 0) {
        
        # 1. Add the 'type' column indicating direction
        res_sig$type <- ifelse(res_sig$logFC > 0, "up", "down")
        
        # 2. Sort descending by logFC (High pos -> Low pos -> Low neg -> Strong neg)
        res_sig <- res_sig[order(res_sig$logFC, decreasing = TRUE), ]
        
        # 3. Reorder columns to display 'type' next to 'logFC'
        res_sig <- res_sig[, c("Gene", "CellType", "Comparison", "type", "logFC", "logCPM", "F", "PValue", "FDR", 
                               paste0("Observations_", cond), paste0("Observations_", ref_condition))]
        
        # 4. Save to the specific comparison folder
        comp_sig_dir <- file.path(out_sig, comp_folder_name)
        dir.create(comp_sig_dir, recursive = TRUE, showWarnings = FALSE)
        
        sig_filename <- paste0("edgeR_Sig_", celltype, "_", comp_folder_name, ".csv")
        write.csv(res_sig, file.path(comp_sig_dir, sig_filename), row.names = FALSE)
        cat("     -> Found", nrow(res_sig), "significant DEGs (sorted).\n")
      } else {
        cat("     -> No significant DEGs passed the threshold (FDR < 0.05, |log2FC| >= 0.5).\n")
      }
      
    } else {
      cat("     SKIP: Insufficient samples. Ref (", n_ref, "), Cond (", n_cond, ") - Both must be > 2.\n")
    }
  }
}

cat("\nAll edgeR pipelines complete! Check the results/edgeR_raw and results/edgeR_significant folders.\n")

Loading required package: limma


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union





PROCESSING CELL TYPE: CM 
  -> Comparing: 6month-protocol vs 3month-protocol 
     -> Found 325 significant DEGs (sorted).
  -> Comparing: Prematurely-terminated vs 3month-protocol 
     -> Found 417 significant DEGs (sorted).
  -> Comparing: PCMV/PRV vs 3month-protocol 
     -> Found 755 significant DEGs (sorted).
  -> Comparing: Rejection vs 3month-protocol 
     -> Found 113 significant DEGs (sorted).

PROCESSING CELL TYPE: EC_papio 
  -> Comparing: 6month-protocol vs 3month-protocol 
     -> No significant DEGs passed the threshold (FDR < 0.05, |log2FC| >= 0.5).
  -> Comparing: Prematurely-terminated vs 3month-protocol 
     SKIP: Insufficient samples. Ref ( 4 ), Cond ( 2 ) - Both must be > 2.
  -> Comparing: PCMV/PRV vs 3month-protocol 
     SKIP: Insufficient samples. Ref ( 4 ), Cond ( 0 ) - Both must be > 2.
  -> Comparing: Rejection vs 3month-protocol 
     SKIP: Insufficient samples. Ref ( 4 ), Cond ( 2 ) - Both must be > 2.

PROCESSING CELL TYPE: EC 
  -> Comparing: 6month-p

# overall barplot

In [2]:
# DEG summary
# Aggregates edgeR significant results and plots Up/Down barplots

options(warn = -1)
options(stringsAsFactors = FALSE)

# Load required packages
quiet_library <- function(pkg) {
  suppressWarnings(suppressMessages(require(pkg, character.only = TRUE)))
}

pkgs <- c("dplyr", "ggplot2", "tidyr", "stringr")
invisible(lapply(pkgs, quiet_library))


# 1. Configuration

# Input directory where the significant CSVs are stored
sig_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/edgeR_significant"

# Output directory for the master CSV and barplots
out_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/overall_barplots"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# Custom plot theme (Arial/sans, Size 5, Bold, No Grids)
custom_theme <- theme(
  text = element_text(size = 5, face = "bold", family = "sans", color = "black"),
  axis.text.x = element_text(angle = 45, hjust = 1, face = "bold", size = 5, family = "sans", color = "black"),
  axis.text.y = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  plot.title = element_text(face = "bold", size = 5, family = "sans", color = "black", hjust = 0.5),
  legend.text = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  legend.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  panel.background = element_blank(),
  panel.grid.major = element_blank(),
  panel.grid.minor = element_blank(),
  axis.line = element_line(color = "black", linewidth = 0.3)
)


# 2. Aggregate signifcant DEGs

cat("Scanning for significant DEG CSVs...\n")

# Recursively find all CSVs in the edgeR_significant directory
sig_files <- list.files(sig_dir, pattern = "\\.csv$", recursive = TRUE, full.names = TRUE)

if (length(sig_files) == 0) {
  stop("No significant CSV files found in the specified directory.")
}

summary_list <- list()

for (f in sig_files) {
  df <- read.csv(f, stringsAsFactors = FALSE)
  
  if (nrow(df) == 0) next
  
  # Extract comparison and cell type from the dataframe
  comp  <- df$Comparison[1]
  ctype <- df$CellType[1]
  
  # Count occurrences
  n_total <- nrow(df)
  n_up    <- sum(df$type == "up", na.rm = TRUE)
  n_down  <- sum(df$type == "down", na.rm = TRUE)
  
  summary_list[[length(summary_list) + 1]] <- data.frame(
    Comparison = comp,
    Cell_Type = ctype,
    Total_Significant_DEGs = n_total,
    Up_DEGs = n_up,
    Down_DEGs = n_down,
    stringsAsFactors = FALSE
  )
}

# Combine all into a master dataframe
master_df <- dplyr::bind_rows(summary_list)

# Save Master CSV
master_csv_path <- file.path(out_dir, "WholeMaster_DEG_Counts.csv")
write.csv(master_df, master_csv_path, row.names = FALSE)
cat("Master CSV saved to:", master_csv_path, "\n")

# ==============================================================================
# 3. Creating the barplots
# ==============================================================================
cat("Generating barplots...\n")

# Convert dataframe to long format for plotting (Up and Down side-by-side)
plot_df <- master_df %>%
  tidyr::pivot_longer(cols = c("Up_DEGs", "Down_DEGs"), 
                      names_to = "Direction", 
                      values_to = "Count") %>%
  dplyr::mutate(Direction = ifelse(Direction == "Up_DEGs", "Upregulated", "Downregulated"))

# Ensure upregulated plots first (left), then downregulated (right)
plot_df$Direction <- factor(plot_df$Direction, levels = c("Upregulated", "Downregulated"))

unique_comps <- unique(plot_df$Comparison)

for (comp in unique_comps) {
  # Subset data for the specific comparison
  sub_df <- plot_df %>% filter(Comparison == comp)
  
  # Plot
  p <- ggplot(sub_df, aes(x = Cell_Type, y = Count, fill = Direction)) +
    geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.7, color = "black", linewidth = 0.15) +
    scale_fill_manual(values = c("Upregulated" = "salmon", "Downregulated" = "skyblue")) +
    # Add text labels right above the bars
    geom_text(aes(label = Count), 
              position = position_dodge(width = 0.8), 
              vjust = -0.5, 
              size = 5 / .pt, fontface = "bold", family = "sans", color = "black") +
    labs(title = comp, x = "Cell Type", y = "Number of Significant DEGs", fill = "Regulation") +
    scale_y_continuous(expand = expansion(mult = c(0, 0.2))) + # Extra room at the top for labels
    custom_theme
  
  # Generate safe filename
  safe_comp_name <- gsub("[^A-Za-z0-9_]", "_", comp)
  plot_filename <- file.path(out_dir, paste0("DEG_Barplot_", safe_comp_name, ".pdf"))
  
  # Save to PDF
  ggsave(plot_filename, plot = p, width = 4, height = 2.5, units = "in", dpi = 300)
  cat("  -> Saved plot for:", comp, "\n")
}

cat("All summaries and barplots completed successfully!\n")

Scanning for significant DEG CSVs...
Master CSV saved to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/overall_barplots/WholeMaster_DEG_Counts.csv 
Generating barplots...
  -> Saved plot for: 6month-protocol vs 3month-protocol 
  -> Saved plot for: PCMV/PRV vs 3month-protocol 
  -> Saved plot for: Prematurely-terminated vs 3month-protocol 
  -> Saved plot for: Rejection vs 3month-protocol 
All summaries and barplots completed successfully!


# Volcano plots: Unlabeled

In [3]:
# VOLCANO PLOT GENERATOR (UNNAMED / NO LABELS)
# Uses edgeR raw outputs to plot the full distribution of DEGs

options(warn = -1)
options(stringsAsFactors = FALSE)

quiet_library <- function(pkg) {
  suppressWarnings(suppressMessages(require(pkg, character.only = TRUE)))
}

pkgs <- c("dplyr", "ggplot2", "stringr")
invisible(lapply(pkgs, quiet_library))

# 1. CONFIGURATION
# Directory containing the raw edgeR outputs (needed for the full background of genes)
raw_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/edgeR_raw"

# Base output directory for the volcano plots
out_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/volcano_unnamed"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# Thresholds 
fdr_threshold   <- 0.05
logfc_threshold <- 0.5

# Custom theme (Arial/sans, Size 5, Bold, No Grids)
custom_theme <- theme(
  text = element_text(size = 5, face = "bold", family = "sans", color = "black"),
  axis.text.x = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.text.y = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  plot.title = element_text(face = "bold", size = 5, family = "sans", color = "black", hjust = 0.5),
  legend.text = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  legend.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  panel.background = element_blank(),
  panel.grid.major = element_blank(),
  panel.grid.minor = element_blank(),
  axis.line = element_line(color = "black", linewidth = 0.3)
)

# 2. GENERATE VOLCANO PLOTS
cat("Scanning for raw edgeR CSVs...\n")

# Find all raw CSV files (recursive ensures we look inside the comparison subfolders)
raw_files <- list.files(raw_dir, pattern = "\\.csv$", recursive = TRUE, full.names = TRUE)

if (length(raw_files) == 0) {
  stop("No raw CSV files found in the specified directory.")
}

for (f in raw_files) {
  df <- read.csv(f, stringsAsFactors = FALSE)
  
  if (nrow(df) == 0) next
  
  comp  <- df$Comparison[1]
  ctype <- df$CellType[1]
  
  # Create safe folder and file names
  safe_comp <- gsub("[^A-Za-z0-9_]", "_", comp)
  safe_ctype <- gsub("[^A-Za-z0-9_]", "_", ctype)
  
  # Create a specific subfolder for this comparison
  comp_out_dir <- file.path(out_dir, safe_comp)
  dir.create(comp_out_dir, recursive = TRUE, showWarnings = FALSE)
  
  # 1. Handle extreme P-values/FDRs (prevent infinity when taking log10)
  # If FDR is exactly 0, replace it with the smallest non-zero FDR in the dataset, or a tiny number
  min_fdr <- min(df$FDR[df$FDR > 0], na.rm = TRUE)
  if (is.na(min_fdr) || min_fdr == Inf) min_fdr <- 1e-300
  df$FDR[df$FDR == 0] <- min_fdr
  
  # Calculate -log10(FDR) for the Y-axis
  df$neg_log10_fdr <- -log10(df$FDR)
  
  # 2. Define Significance Categories
  df <- df %>%
    mutate(
      Significance = case_when(
        FDR < fdr_threshold & logFC >= logfc_threshold ~ "Upregulated",
        FDR < fdr_threshold & logFC <= -logfc_threshold ~ "Downregulated",
        TRUE ~ "Not Significant"
      )
    )
  
  # Reorder factor so "Not Significant" is plotted first (in the background)
  df$Significance <- factor(df$Significance, levels = c("Not Significant", "Downregulated", "Upregulated"))
  
  # 3. Create the Volcano Plot
  p <- ggplot(df, aes(x = logFC, y = neg_log10_fdr, color = Significance)) +
    geom_point(alpha = 0.8, size = 0.5, stroke = 0) +
    # Changed skyblue to royalblue for a darker, deeper blue
    scale_color_manual(values = c("Not Significant" = "grey80", 
                                  "Downregulated" = "royalblue", 
                                  "Upregulated" = "salmon")) +
    # Add threshold lines
    geom_vline(xintercept = c(-logfc_threshold, logfc_threshold), linetype = "dashed", color = "black", linewidth = 0.2) +
    geom_hline(yintercept = -log10(fdr_threshold), linetype = "dashed", color = "black", linewidth = 0.2) +
    labs(
      title = paste0(ctype, " (", comp, ")"),
      x = bquote(bold(log[2]("Fold Change"))),
      y = bquote(bold(-log[10]("FDR"))),
      color = "Regulation"
    ) +
    custom_theme +
    # Override legend point size so it's visible even though plot points are tiny
    guides(color = guide_legend(override.aes = list(size = 2)))
  
  # ---------------------------------------------------------------------------
  # FULL FRAME TOGGLE:
  # To add a full box frame around the plot, simply uncomment the '#' at the 
  # start of the line below. This overrides the default axis lines.
  # ---------------------------------------------------------------------------
  p <- p + theme(panel.border = element_rect(colour = "black", fill = NA, linewidth = 0.3), axis.line = element_blank())

  # 4. Save the plot into the comparison-specific subfolder
  plot_filename <- file.path(comp_out_dir, paste0("Volcano_", safe_ctype, "_", safe_comp, ".pdf"))
  
  ggsave(plot_filename, plot = p, width = 2.75, height = 2.25, units = "in", dpi = 300)
  cat("  -> Saved volcano plot for:", ctype, "in folder:", safe_comp, "\n")
}

cat("\nAll volcano plots generated and safely organized into comparison subfolders at:\n", out_dir, "\n")

Scanning for raw edgeR CSVs...
  -> Saved volcano plot for: CM in folder: 6month_protocol_vs_3month_protocol 
  -> Saved volcano plot for: EC in folder: 6month_protocol_vs_3month_protocol 
  -> Saved volcano plot for: EC_papio in folder: 6month_protocol_vs_3month_protocol 
  -> Saved volcano plot for: FB in folder: 6month_protocol_vs_3month_protocol 
  -> Saved volcano plot for: Lymph_papio in folder: 6month_protocol_vs_3month_protocol 
  -> Saved volcano plot for: MC in folder: 6month_protocol_vs_3month_protocol 
  -> Saved volcano plot for: Mye_papio in folder: 6month_protocol_vs_3month_protocol 
  -> Saved volcano plot for: NC in folder: 6month_protocol_vs_3month_protocol 
  -> Saved volcano plot for: CM in folder: PCMV_PRV_vs_3month_protocol 
  -> Saved volcano plot for: EC in folder: PCMV_PRV_vs_3month_protocol 
  -> Saved volcano plot for: FB in folder: PCMV_PRV_vs_3month_protocol 
  -> Saved volcano plot for: Lymph_papio in folder: PCMV_PRV_vs_3month_protocol 
  -> Saved volcano

# Volcano plots: Labelled with chosen genes

In [5]:
# ==============================================================================
# PROFESSIONAL TARGETED VOLCANO PLOT GENERATOR 
# Highlights specific genes of interest with a comprehensive color legend.
# Designed for high replicability across diverse projects and comparisons.
# ==============================================================================

options(warn = -1)
options(stringsAsFactors = FALSE)

# Safely load required libraries
quiet_library <- function(pkg) {
  suppressWarnings(suppressMessages(require(pkg, character.only = TRUE)))
}

pkgs <- c("dplyr", "ggplot2", "stringr", "RColorBrewer")
invisible(lapply(pkgs, quiet_library))

# 1. Configurations

# A. Input/Output paths

base_raw_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/edgeR_raw"

# Output directory where the final volcano plots should be saved
out_dir      <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/volcano_named"

# B. Experiment details

comp_name        <- "PCMV_PRV_vs_3month-protocol"

# The specific cell type to extract and plot
target_cell_type <- "EC"

# --- C. Statistical thresholds ---
fdr_threshold   <- 0.05
logfc_threshold <- 0.5    

# --- D. Genes to Highlight ---
# Add or remove any gene names here. The script will automatically skip missing ones.
genes_to_highlight <- c(
  "Sus_scrofa___MX1", "Sus_scrofa___MX2", "Sus_scrofa___ISG20", "Sus_scrofa___CXCL10", 
  "Sus_scrofa___IFIT1", "Sus_scrofa___OASL", 
  "Sus_scrofa___TSPAN18", "Sus_scrofa___VWF", "Sus_scrofa___SERPINE1"
)

# 2. DATA PREPARATION & VALIDATION
# Create safe directory and file names

safe_comp  <- gsub("[^A-Za-z0-9_\\-]", "_", comp_name) # Preserves hyphens
safe_ctype <- gsub("[^A-Za-z0-9_]", "_", target_cell_type)

comp_out_dir <- file.path(out_dir, comp_name) # Retains exact comparison name for folder
dir.create(comp_out_dir, recursive = TRUE, showWarnings = FALSE)

# Construct expected raw edgeR CSV path
raw_file_path <- file.path(base_raw_dir, comp_name, paste0("edgeR_Raw_", safe_ctype, "_", safe_comp, ".csv"))

if (!file.exists(raw_file_path)) {
  stop(paste("ERROR: Cannot find raw CSV file at:\n", raw_file_path, 
             "\nPlease verify your 'base_raw_dir', 'comp_name', and 'target_cell_type'."))
}

df <- read.csv(raw_file_path, stringsAsFactors = FALSE)

# Validate which highlight genes actually exist in the current dataset
actual_highlights <- intersect(genes_to_highlight, df$Gene)
missing_genes <- setdiff(genes_to_highlight, df$Gene)

if(length(missing_genes) > 0) {
  cat("WARNING: The following requested genes were NOT found in this dataset and will be skipped:\n", 
      paste(missing_genes, collapse = ", "), "\n\n")
}

# Handle extreme P-values/FDRs (prevents 'infinity' when taking -log10)
min_fdr <- min(df$FDR[df$FDR > 0], na.rm = TRUE)
if (is.na(min_fdr) || min_fdr == Inf) min_fdr <- 1e-300
df$FDR[df$FDR == 0] <- min_fdr
df$neg_log10_fdr <- -log10(df$FDR)

# Categorize background genes by significance thresholds
df <- df %>%
  mutate(
    Significance = case_when(
      FDR < fdr_threshold & logFC >= logfc_threshold ~ "Upregulated",
      FDR < fdr_threshold & logFC <= -logfc_threshold ~ "Downregulated",
      TRUE ~ "Not Significant"
    )
  )

# Enforce plotting order (Not Significant drawn first in background)
df$Significance <- factor(df$Significance, levels = c("Not Significant", "Downregulated", "Upregulated"))

# Split dataframe: background vs foreground (highlighted)
df_bg <- df[!df$Gene %in% actual_highlights, ]
df_fg <- df[df$Gene %in% actual_highlights, ]

# Order foreground legend alphabetically
if(nrow(df_fg) > 0) {
  df_fg$Gene <- factor(df_fg$Gene, levels = sort(unique(df_fg$Gene)))
}

# 3. VISUALIZATION THEME & PALETTES

# Highly structured, grid-free publication theme (Arial, Size 5, Bold)
custom_theme <- theme(
  text = element_text(size = 5, face = "bold", family = "sans", color = "black"),
  axis.text.x = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.text.y = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  plot.title = element_text(face = "bold", size = 5, family = "sans", color = "black", hjust = 0.5),
  legend.text = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  legend.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  panel.background = element_blank(),
  panel.grid.major = element_blank(),
  panel.grid.minor = element_blank(),
  panel.border = element_rect(colour = "black", fill = NA, linewidth = 0.3), # Full frame
  axis.line = element_blank(), # Suppress standard axis lines due to full frame
  legend.key.size = unit(0.3, "cm")
)

# Extended Professional Color Palette (Supports up to 50 highlighted genes)
# Combines Set1, Dark2, Paired, and distinct high-contrast hexadecimal colors
prof_colors <- c(
  "#FFD700", # Gold
  "#808080", # Gray
  "#87CEFA", # Light Sky Blue
  "#800080", # Purple
  "#E41A1C", # Red
  "#377EB8", # Blue
  "#4DAF4A", # Green
  "#FF7F00", # Orange
  "#A65628", # Brown
  "#F781BF", # Pink
  "#1B9E77", # Teal
  "#D95F02", # Burnt Orange
  "#7570B3", # Muted Purple
  "#E7298A", # Magenta
  "#66A61E", # Olive Green
  "#E6AB02", # Mustard Yellow
  "#1F78B4", # Medium Blue
  "#B2DF8A", # Light Green
  "#33A02C", # Forest Green
  "#FB9A99", # Light Pink
  "#FDBF6F", # Light Orange
  "#CAB2D6", # Light Purple
  "#6A3D9A", # Dark Purple
  "#FFFF99", # Light Yellow
  "#BEBADA", # Lavender
  "#FB8072", # Salmon
  "#80B1D3", # Sky Blue
  "#B3DE69", # Light Lime
  "#D9D9D9", # Light Gray
  "#BC80BD", # Muted Magenta
  "#000000", # Black
  "#1ce6ff", # Bright Cyan
  "#ff34ff", # Neon Magenta
  "#008941", # Emerald Green
  "#006fa6", # Deep Blue
  "#a30059", # Dark Maroon
  "#7a4900", # Dark Brown
  "#0000a6", # Dark Navy
  "#63ffac", # Neon Mint
  "#b79762", # Tan
  "#004d43"  # Dark Teal
)

# Expand palette dynamically if user supplies >50 genes
if(length(actual_highlights) > length(prof_colors)) {
  prof_colors <- colorRampPalette(prof_colors)(length(actual_highlights))
}

# 4. PLOT GENERATION

p <- ggplot() +
  
  # LAYER 1: Background genes (No borders, mapped to significance)
  geom_point(data = df_bg, aes(x = logFC, y = neg_log10_fdr, color = Significance), 
             alpha = 0.8, size = 0.5, stroke = 0) +
  scale_color_manual(values = c("Not Significant" = "grey80", 
                                "Downregulated" = "royalblue", 
                                "Upregulated" = "salmon")) +
  
  # LAYER 2: Highlighted genes (Shape 21 for fill + black border)
  geom_point(data = df_fg, aes(x = logFC, y = neg_log10_fdr, fill = Gene), 
             shape = 21, color = "black", stroke = 0.2, size = 1.25) +
  scale_fill_manual(values = prof_colors) +
  
  # LAYER 3: Threshold lines
  geom_vline(xintercept = c(-logfc_threshold, logfc_threshold), linetype = "dashed", color = "black", linewidth = 0.2) +
  geom_hline(yintercept = -log10(fdr_threshold), linetype = "dashed", color = "black", linewidth = 0.2) +
  
  # LAYER 4: Formatting & Labels
  labs(
    title = paste0(target_cell_type, " (", comp_name, ")"),
    x = bquote(bold(log[2]("Fold Change"))),
    y = bquote(bold(-log[10]("FDR"))),
    color = "Regulation",
    fill = "Highlighted Genes"
  ) +
  custom_theme +
  
  # Force legend dots to be visually distinct regardless of plot point size
  guides(
    color = guide_legend(override.aes = list(size = 2), order = 1),
    fill  = guide_legend(override.aes = list(size = 2), order = 2)
  )

# 5. EXPORT

plot_filename <- file.path(comp_out_dir, paste0("Volcano_Highlighted_", safe_ctype, "_", safe_comp, ".pdf"))

# Width set to 2.75 inches; expand if legend gets cut off with many genes
ggsave(plot_filename, plot = p, width = 3.40, height = 1.75, units = "in", dpi = 300)

cat("Successfully generated targeted volcano plot!\nSaved to:", plot_filename, "\n")

Successfully generated targeted volcano plot!
Saved to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/volcano_named/PCMV_PRV_vs_3month-protocol/Volcano_Highlighted_EC_PCMV_PRV_vs_3month-protocol.pdf 


# Venn diagrams

In [2]:

# Creating a 4-set venn diagram (REF: 3MONTH-PROTOCOL)
# Compares DEGs across 4 specific conditions vs. 3month-protocol for selected cell_type

# Suppress annoying VennDiagram log files
# Note: 'readr' is strictly excluded to bypass GLIBC OS-level errors
suppressPackageStartupMessages({
  library(VennDiagram)
  library(grid)
  library(dplyr)
  library(stringr)
  futile.logger::flog.threshold(futile.logger::ERROR, name = "VennDiagramLogger")
})

options(stringsAsFactors = FALSE)

# 1. Configs

# Base directory where the significant edgeR CSVs are stored (Ref_3month)
sig_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/edgeR_significant"

# Output directory for the Venn diagrams and data tables
out_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# Define exactly which Cell Types you want to run (Comment out or add as needed)
cell_types_to_plot <- c(
  "FB", 
  "EC", 
  "CM"
)

ref_condition <- "3month-protocol"

# Define the 4 conditions tested against the 3month-protocol
# Order matters: this will map to sets A, B, C, D in the Venn diagram
conditions <- c(
  "6month-protocol",
  "Prematurely-terminated",
  "Rejection",
  "PCMV/PRV"
)

# Short labels for the CSV output regions
short_labels <- c("6mo", "PremTerm", "Rej", "PCMV")

# Colors for the 4 sets (Clean, pastel, publication-ready palette)
venn_colors <- c("#08528f", "#C9B8D9", "#E09E9E", "#F0C096") 
venn_alpha  <- 0.7  # Transparency level

# 2. HELPER FUNCTIONS

#Load genes from the edgeR significant CSVs
read_sig_genes <- function(filepath) {
  if (!file.exists(filepath)) {
    warning(sprintf("     [WARNING] File not found: %s", filepath))
    return(character(0))
  }
  
  # Using base R read.csv to avoid GLIBC errors
  df <- read.csv(filepath, stringsAsFactors = FALSE)
  
  if (!"Gene" %in% names(df)) {
    stop(sprintf("No 'Gene' column in %s", filepath))
  }
  g <- as.character(df$Gene)
  g <- g[!is.na(g) & nzchar(g)]
  return(unique(g))
}

# Calculate all 15 regions for a 4-set Venn (Matrix Approach)
get_exclusive_regions <- function(gene_lists, labels) {
  all_genes <- unique(unlist(gene_lists))
  if (length(all_genes) == 0) return(list())
  
  # Create a presence/absence matrix (Rows = Genes, Cols = Conditions)
  mat <- sapply(gene_lists, function(x) all_genes %in% x)
  rownames(mat) <- all_genes
  colnames(mat) <- labels
  
  # Map each gene to a binary pattern (e.g., "1_0_1_0")
  patterns <- apply(mat, 1, function(row) paste(as.integer(row), collapse=""))
  
  # Group genes by their exact intersection pattern
  raw_regions <- split(all_genes, patterns)
  
  # Rename patterns to meaningful string labels (e.g., "6mo_Rej")
  final_regions <- list()
  for (pat in names(raw_regions)) {
    if (pat == "0000") next 
    
    bool_vec <- as.logical(as.integer(strsplit(pat, "")[[1]]))
    region_name <- paste(labels[bool_vec], collapse="_")
    final_regions[[region_name]] <- raw_regions[[pat]]
  }
  
  return(final_regions)
}

# Plotting code (4-Set)
draw_and_save_venn4 <- function(A, B, C, D, out_pdf, out_png, title) {
  
  v <- draw.quad.venn(
    area1 = length(A), area2 = length(B), area3 = length(C), area4 = length(D),
    n12 = length(intersect(A, B)), n13 = length(intersect(A, C)),
    n14 = length(intersect(A, D)), n23 = length(intersect(B, C)),
    n24 = length(intersect(B, D)), n34 = length(intersect(C, D)),
    n123 = length(Reduce(intersect, list(A, B, C))),
    n124 = length(Reduce(intersect, list(A, B, D))),
    n134 = length(Reduce(intersect, list(A, C, D))),
    n234 = length(Reduce(intersect, list(B, C, D))),
    n1234 = length(Reduce(intersect, list(A, B, C, D))),
    
    category = c("", "", "", ""),
    fill = venn_colors,
    alpha = rep(venn_alpha, 4),
    lty = "blank",               
    cex = 1.6,                    # Font size for the intersection numbers
    fontface = "bold",            # Bold text
    fontfamily = "sans",
    cat.cex = 1.3,
    cat.pos = c(330, 40, 140, 220),
    cat.dist = c(0.1, 0.1, 0.1, 0.1),
    ind = FALSE                   # Suppress default internal drawing
  )
  
  # Save to PDF
  pdf(out_pdf, width = 8.5, height = 7.5)
  grid.newpage()
  grid.draw(v)
  grid.text(title, y = unit(0.95, "npc"), gp = gpar(fontsize = 16, fontface = "bold", fontfamily="sans"))
  dev.off()
  
  # Save to PNG (300 DPI)
  png(out_png, width = 2550, height = 2250, res = 300)
  grid.newpage()
  grid.draw(v)
  grid.text(title, y = unit(0.95, "npc"), gp = gpar(fontsize = 16, fontface = "bold", fontfamily="sans"))
  dev.off()
}


# 3. The main loop

if (length(conditions) != 4) {
  stop("ERROR: You must define exactly 4 conditions for a quad-Venn diagram.")
}

cat("\n=========================================================\n")
cat("STARTING 4-SET VENN DIAGRAM GENERATION\n")
cat("=========================================================\n")

for (ctype in cell_types_to_plot) {
  cat("\nProcessing Cell Type:", ctype, "\n")
  
  # 1. Dynamically construct file paths matching edgeR sanitization logic
  paths <- sapply(conditions, function(cond) {

    safe_cond <- gsub("[^A-Za-z0-9_]", "_", cond)
    comp_folder_name <- paste0(safe_cond, "_vs_", ref_condition)
    
    file.path(sig_dir, comp_folder_name, paste0("edgeR_Sig_", ctype, "_", comp_folder_name, ".csv"))
  })
  
  # 2. Extract significant genes
  cat("  -> Loading genes...\n")
  genes_A <- read_sig_genes(paths[1])
  genes_B <- read_sig_genes(paths[2])
  genes_C <- read_sig_genes(paths[3])
  genes_D <- read_sig_genes(paths[4])
  
  total_loaded <- length(genes_A) + length(genes_B) + length(genes_C) + length(genes_D)
  if (total_loaded == 0) {
    cat("  -> Skipping", ctype, "(No significant genes found across any comparison).\n")
    next
  }
  
  # 3. Calculate intersections (Matrix approach)
  gene_lists_combined <- list(genes_A, genes_B, genes_C, genes_D)
  regions <- get_exclusive_regions(gene_lists_combined, short_labels)
  
  # 4. Export detailed CSV (One clean master file)
  summary_df <- data.frame(
    Region = names(regions),
    Gene_Count = vapply(regions, length, integer(1)),
    Genes_List = vapply(regions, function(x) paste(sort(unique(x)), collapse = ", "), character(1))
  )
  
  csv_out <- file.path(out_dir, paste0("Venn_Data_Summary_", ctype, ".csv"))
  write.csv(summary_df, csv_out, row.names = FALSE)
  cat("  -> Saved region mapping to:", csv_out, "\n")
  
  # 5. Draw and save plots
  title_str <- paste(ctype, "Significant DEGs vs 3month-protocol")
  pdf_out <- file.path(out_dir, paste0("Venn_Diagram_", ctype, ".pdf"))
  png_out <- file.path(out_dir, paste0("Venn_Diagram_", ctype, ".png"))
  
  draw_and_save_venn4(genes_A, genes_B, genes_C, genes_D, pdf_out, png_out, title_str)
  cat("  -> Saved plots to:", pdf_out, "& .png\n")
}

cat("\n===\n")
cat("ALL VENN DIAGRAMS COMPLETED!\n")
cat("Check output directory:", out_dir, "\n")

NULL


STARTING 4-SET VENN DIAGRAM GENERATION

Processing Cell Type: FB 
  -> Loading genes...
  -> Saved region mapping to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Data_Summary_FB.csv 
  -> Saved plots to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Diagram_FB.pdf & .png

Processing Cell Type: EC 
  -> Loading genes...
  -> Saved region mapping to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Data_Summary_EC.csv 
  -> Saved plots to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Diagram_EC.pdf & .png

Processing Cell Type: CM 
  -> Loading genes...
  -> Saved region mapping to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_

In [3]:

# Creating a 4-set venn diagram (REF: 3MONTH-PROTOCOL)
# Compares DEGs across 4 specific conditions vs. 3month-protocol for selected cell_type
# Improved visualization parameters

options(stringsAsFactors = FALSE)

# Package installation & loading
required_pkgs <- c("ggVennDiagram", "ggplot2", "dplyr", "stringr")
for (pkg in required_pkgs) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    message("Installing ", pkg, " ...")
    install.packages(pkg, quiet = TRUE)
  }
}

tryCatch(
  suppressPackageStartupMessages({
    library(ggVennDiagram)
    library(ggplot2)
    library(dplyr)
    library(stringr)
  }),
  error = function(e) {
    stop(
      "ggVennDiagram requires the 'sf' package, which needs system libraries.\n",
      "  Ubuntu/Debian : sudo apt install libgdal-dev libgeos-dev libproj-dev\n",
      "  CentOS/RHEL   : sudo yum install gdal-devel geos-devel proj-devel\n",
      "Then re-run this script.\n\nOriginal error: ", conditionMessage(e)
    )
  }
)


# Configs

# A. Input directory (edgeR significant results, Ref_3month) 
sig_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/edgeR_significant"

#B. Output directory 
out_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# C. Reference condition
ref_condition <- "3month-protocol"

# D. Cell types to process ---
cell_types_to_plot <- c("FB", "EC", "CM")

# E. Conditions (must be exactly 4 for a quad Venn)
conditions <- c(
  "6month-protocol",
  "Prematurely-terminated",
  "Rejection",
  "PCMV/PRV"
)

# Short labels used as set names inside the diagram and in the CSV
short_labels <- c("6mo", "PremTerm", "Rej", "PCMV")

# --- F. Colours ---
label_colors <- c("#08528f", "#C9B8D9", "#E09E9E", "#F0C096")

# Edge (border) colours for each ellipse
edge_colors <- c("#08528f", "#C9B8D9", "#E09E9E", "#F0C096")

# Region-fill gradient 
fill_low  <- "#FFFFFF"    # white / near-white for sparse regions
fill_high <- "#B22222"    # Dark-bright red for dense regions

# --- G. Typography & sizing ---
count_label_size    <- 4.0   # Size of count numbers inside each region
count_label_color   <- "black"
category_label_size <- 4.5   # Size of set-name labels around the perimeter
edge_line_size      <- 0.6   # Thickness of ellipse borders
plot_width_in       <- 8.5   # PDF/PNG width  (inches)
plot_height_in      <- 7.5   # PDF/PNG height (inches)
png_dpi             <- 300


# 2. helper functions

# Load significant genes from an edgeR CSV
read_sig_genes <- function(filepath) {
  if (!file.exists(filepath)) {
    warning(sprintf("  [WARNING] File not found: %s", filepath))
    return(character(0))
  }
  df <- read.csv(filepath, stringsAsFactors = FALSE)
  if (!"Gene" %in% names(df)) {
    stop(sprintf("No 'Gene' column in %s", filepath))
  }
  g <- as.character(df$Gene)
  unique(g[!is.na(g) & nzchar(g)])
}

#Calculate all 15 exclusive regions (for CSV export)
get_exclusive_regions <- function(gene_lists, labels) {
  all_genes <- unique(unlist(gene_lists))
  if (length(all_genes) == 0) return(list())

  mat <- sapply(gene_lists, function(x) all_genes %in% x)
  rownames(mat) <- all_genes
  colnames(mat) <- labels

  patterns <- apply(mat, 1, function(row) paste(as.integer(row), collapse = ""))
  raw_regions <- split(all_genes, patterns)

  final <- list()
  for (pat in names(raw_regions)) {
    if (pat == "0000") next
    bool_vec <- as.logical(as.integer(strsplit(pat, "")[[1]]))
    region_name <- paste(labels[bool_vec], collapse = "_")
    final[[region_name]] <- raw_regions[[pat]]
  }
  final
}


# 3. Main loop

if (length(conditions) != 4) {
  stop("ERROR: You must define exactly 4 conditions for a quad-Venn diagram.")
}

cat("\n==========\n")
cat("STARTING 4-SET VENN DIAGRAM GENERATION (ggVennDiagram)\n")
cat("Reference:", ref_condition, "\n")
cat("===============\n")

for (ctype in cell_types_to_plot) {

  cat("\nProcessing Cell Type:", ctype, "\n")

  # ---- 3A. Build file paths ----
  paths <- sapply(conditions, function(cond) {
    safe_cond <- gsub("[^A-Za-z0-9_]", "_", cond)
    comp_folder <- paste0(safe_cond, "_vs_", ref_condition)
    file.path(sig_dir, comp_folder,
              paste0("edgeR_Sig_", ctype, "_", comp_folder, ".csv"))
  })

  # ---- 3B. Load significant genes ----
  cat("  -> Loading genes...\n")
  gene_vectors <- lapply(paths, read_sig_genes)
  names(gene_vectors) <- short_labels

  total_loaded <- sum(sapply(gene_vectors, length))
  if (total_loaded == 0) {
    cat("  -> Skipping", ctype, "(no significant genes in any comparison).\n")
    next
  }

  for (i in seq_along(short_labels)) {
    cat("     ", short_labels[i], ":", length(gene_vectors[[i]]), "genes\n")
  }

  # 3C. Region summary CSV 
  regions <- get_exclusive_regions(gene_vectors, short_labels)

  summary_df <- data.frame(
    Region     = names(regions),
    Gene_Count = vapply(regions, length, integer(1)),
    Genes_List = vapply(regions, function(x) {
      paste(sort(unique(x)), collapse = ", ")
    }, character(1))
  )

  csv_out <- file.path(out_dir, paste0("Venn_Data_Summary_", ctype, ".csv"))
  write.csv(summary_df, csv_out, row.names = FALSE)
  cat("  -> Saved region summary:", csv_out, "\n")

  # 3D. Build ggVennDiagram plot 
  title_str <- paste0(ctype, " — Significant DEGs vs ", ref_condition)

  p <- ggVennDiagram(
    gene_vectors,
    label       = "count",
    label_alpha = 0,
    label_size  = count_label_size,
    label_color = count_label_color,
    edge_size   = edge_line_size,
    set_size    = category_label_size,
    set_color   = label_colors
  ) +

  # Region fill gradient
  scale_fill_gradient(
    low  = fill_low,
    high = fill_high,
    name = "Gene count"
  ) +

  # Edge colours
  scale_color_manual(
    values = setNames(edge_colors, as.character(seq_along(edge_colors))),
    guide  = "none"
  ) +

  # Padding so perimeter labels don't clip
  scale_x_continuous(expand = expansion(mult = 0.15)) +
  scale_y_continuous(expand = expansion(mult = 0.15)) +

  labs(title = title_str) +
  theme_void() +
  theme(
    plot.title      = element_text(hjust = 0.5, face = "bold",
                                   size = 12, family = "sans"),
    legend.position = "none"
  )

  #3E. Save PDF (transparent background)
  pdf_out <- file.path(out_dir, paste0("Venn_Diagram_", ctype, ".pdf"))
  ggsave(pdf_out, plot = p,
         width  = plot_width_in,
         height = plot_height_in,
         bg     = "transparent")
  cat("  -> Saved PDF:", pdf_out, "\n")

  # 3F. Save PNG
  png_out <- file.path(out_dir, paste0("Venn_Diagram_", ctype, ".png"))
  ggsave(png_out, plot = p,
         width  = plot_width_in,
         height = plot_height_in,
         dpi    = png_dpi,
         bg     = "white")
  cat("  -> Saved PNG:", png_out, "\n")
}

cat("\n=========================================================\n")
cat("ALL 4-SET VENN DIAGRAMS COMPLETED!\n")
cat("Output directory:", out_dir, "\n")
cat("=========================================================\n")


STARTING 4-SET VENN DIAGRAM GENERATION (ggVennDiagram)
Reference: 3month-protocol 

Processing Cell Type: FB 
  -> Loading genes...
      6mo : 161 genes
      PremTerm : 741 genes
      Rej : 252 genes
      PCMV : 292 genes
  -> Saved region summary: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Data_Summary_FB.csv 
  -> Saved PDF: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Diagram_FB.pdf 
  -> Saved PNG: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Diagram_FB.png 

Processing Cell Type: EC 
  -> Loading genes...
      6mo : 265 genes
      PremTerm : 1149 genes
      Rej : 188 genes
      PCMV : 1116 genes
  -> Saved region summary: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Data_Summary_

In [5]:

# Creating a 4-set venn diagram (REF: 3MONTH-PROTOCOL)
# Compares DEGs across 4 specific conditions vs. 3month-protocol for selected cell_type

options(stringsAsFactors = FALSE)

# --- Package installation & loading ---
required_pkgs <- c("ggVennDiagram", "ggplot2", "dplyr", "stringr")
for (pkg in required_pkgs) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    message("Installing ", pkg, " ...")
    install.packages(pkg, quiet = TRUE)
  }
}

tryCatch(
  suppressPackageStartupMessages({
    library(ggVennDiagram)
    library(ggplot2)
    library(dplyr)
    library(stringr)
  }),
  error = function(e) {
    stop(
      "ggVennDiagram requires the 'sf' package, which needs system libraries.\n",
      "  Ubuntu/Debian : sudo apt install libgdal-dev libgeos-dev libproj-dev\n",
      "  CentOS/RHEL   : sudo yum install gdal-devel geos-devel proj-devel\n",
      "Then re-run this script.\n\nOriginal error: ", conditionMessage(e)
    )
  }
)

# 1. Configs

# --- A. Input directory (edgeR significant results, Ref_3month) ---
sig_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/edgeR_significant"

# --- B. Output directory ---
out_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# --- C. Reference condition ---
ref_condition <- "3month-protocol"

# --- D. Cell types to process ---
cell_types_to_plot <- c("FB", "EC", "CM")

# --- E. Conditions (must be exactly 4 for a quad Venn) ---
conditions <- c(
  "6month-protocol",
  "Prematurely-terminated",
  "Rejection",
  "PCMV/PRV"
)

# Short labels used as set names inside the diagram and in the CSV
short_labels <- c("6mo", "PremTerm", "Rej", "PCMV")

# --- F. Colours ---
# Category-label colours
label_colors <- c("#08528f", "#C9B8D9", "#E09E9E", "#F0C096")

# Edge (border) colours for each ellipse
edge_colors <- c("#08528f", "#C9B8D9", "#E09E9E", "#F0C096")

# Region-fill gradient
fill_low  <- "#FFFFFF"    # white / near-white for sparse regions
fill_high <- "#B22222"    # firebrick red for dense regions

# --- G. Ellipse border thickness ---
# Controls how thick the ellipse outlines are drawn.
# *** Change this single value to adjust all borders at once. ***
#   0.3  = hairline
#   0.6  = subtle
#   1.0  = medium
#   1.5  = visible
#   2.0  = thick   (recommended for publication)
#   2.5  = heavy
#   3.0+ = very bold
edge_thickness <- 4.0

# --- H. Typography & sizing ---
count_label_size    <- 4.0   # Size of count numbers inside each region
count_label_color   <- "black"
category_label_size <- 5   # Size of set-name labels around the perimeter
plot_width_in       <- 8.5   # PDF/PNG width  (inches)
plot_height_in      <- 7.5   # PDF/PNG height (inches)
png_dpi             <- 300


# 2. helper functions

# ---- Load significant genes from an edgeR CSV ----
read_sig_genes <- function(filepath) {
  if (!file.exists(filepath)) {
    warning(sprintf("  [WARNING] File not found: %s", filepath))
    return(character(0))
  }
  df <- read.csv(filepath, stringsAsFactors = FALSE)
  if (!"Gene" %in% names(df)) {
    stop(sprintf("No 'Gene' column in %s", filepath))
  }
  g <- as.character(df$Gene)
  unique(g[!is.na(g) & nzchar(g)])
}

# Calculate all 15 exclusive regions (for CSV export) 
get_exclusive_regions <- function(gene_lists, labels) {
  all_genes <- unique(unlist(gene_lists))
  if (length(all_genes) == 0) return(list())

  mat <- sapply(gene_lists, function(x) all_genes %in% x)
  rownames(mat) <- all_genes
  colnames(mat) <- labels

  patterns <- apply(mat, 1, function(row) paste(as.integer(row), collapse = ""))
  raw_regions <- split(all_genes, patterns)

  final <- list()
  for (pat in names(raw_regions)) {
    if (pat == "0000") next
    bool_vec <- as.logical(as.integer(strsplit(pat, "")[[1]]))
    region_name <- paste(labels[bool_vec], collapse = "_")
    final[[region_name]] <- raw_regions[[pat]]
  }
  final
}


# 3. Main loops

if (length(conditions) != 4) {
  stop("ERROR:  must define exactly 4 conditions for a quad-Venn diagram.")
}

cat("\n=========================================================\n")
cat("STARTING 4-SET VENN DIAGRAM GENERATION (ggVennDiagram)\n")
cat("Reference:", ref_condition, "\n")
cat("=========================================================\n")

for (ctype in cell_types_to_plot) {

  cat("\nProcessing Cell Type:", ctype, "\n")

  # ---- 3A. Build file paths ----
  paths <- sapply(conditions, function(cond) {
    safe_cond <- gsub("[^A-Za-z0-9_]", "_", cond)
    comp_folder <- paste0(safe_cond, "_vs_", ref_condition)
    file.path(sig_dir, comp_folder,
              paste0("edgeR_Sig_", ctype, "_", comp_folder, ".csv"))
  })

  # ---- 3B. Load significant genes ----
  cat("  -> Loading genes...\n")
  gene_vectors <- lapply(paths, read_sig_genes)
  names(gene_vectors) <- short_labels

  total_loaded <- sum(sapply(gene_vectors, length))
  if (total_loaded == 0) {
    cat("  -> Skipping", ctype, "(no significant genes in any comparison).\n")
    next
  }

  for (i in seq_along(short_labels)) {
    cat("     ", short_labels[i], ":", length(gene_vectors[[i]]), "genes\n")
  }

  # ---- 3C. Region summary CSV ----
  regions <- get_exclusive_regions(gene_vectors, short_labels)

  summary_df <- data.frame(
    Region     = names(regions),
    Gene_Count = vapply(regions, length, integer(1)),
    Genes_List = vapply(regions, function(x) {
      paste(sort(unique(x)), collapse = ", ")
    }, character(1))
  )

  csv_out <- file.path(out_dir, paste0("Venn_Data_Summary_", ctype, ".csv"))
  write.csv(summary_df, csv_out, row.names = FALSE)
  cat("  -> Saved region summary:", csv_out, "\n")

  # ---- 3D. Build ggVennDiagram plot ----
  title_str <- paste0(ctype, " — Significant DEGs vs ", ref_condition)

  p <- ggVennDiagram(
    gene_vectors,
    label       = "count",
    label_alpha = 0,
    label_size  = count_label_size,
    label_color = count_label_color,
    edge_size   = edge_thickness,      # <-- configured in Section 1G
    set_size    = category_label_size,
    set_color   = label_colors
  ) +

  # Region fill gradient
  scale_fill_gradient(
    low  = fill_low,
    high = fill_high,
    name = "Gene count"
  ) +

  # Edge colours
  scale_color_manual(
    values = setNames(edge_colors, as.character(seq_along(edge_colors))),
    guide  = "none"
  ) +

  # Padding so perimeter labels don't clip
  scale_x_continuous(expand = expansion(mult = 0.15)) +
  scale_y_continuous(expand = expansion(mult = 0.15)) +

  labs(title = title_str) +
  theme_void() +
  theme(
    plot.title      = element_text(hjust = 0.5, face = "bold",
                                   size = 12, family = "sans"),
    legend.position = "none"
  )

  # 3E. Save PDF (transparent background)
  pdf_out <- file.path(out_dir, paste0("Venn_Diagram_", ctype, ".pdf"))
  ggsave(pdf_out, plot = p,
         width  = plot_width_in,
         height = plot_height_in,
         bg     = "transparent")
  cat("  -> Saved PDF:", pdf_out, "\n")

  # 3F. Save PNG 
  png_out <- file.path(out_dir, paste0("Venn_Diagram_", ctype, ".png"))
  ggsave(png_out, plot = p,
         width  = plot_width_in,
         height = plot_height_in,
         dpi    = png_dpi,
         bg     = "white")
  cat("  -> Saved PNG:", png_out, "\n")
}

cat("\n=========================================================\n")
cat("ALL 4-SET VENN DIAGRAMS COMPLETED!\n")
cat("Output directory:", out_dir, "\n")
cat("=========================================================\n")


STARTING 4-SET VENN DIAGRAM GENERATION (ggVennDiagram)
Reference: 3month-protocol 

Processing Cell Type: FB 
  -> Loading genes...
      6mo : 161 genes
      PremTerm : 741 genes
      Rej : 252 genes
      PCMV : 292 genes
  -> Saved region summary: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Data_Summary_FB.csv 
  -> Saved PDF: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Diagram_FB.pdf 
  -> Saved PNG: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Diagram_FB.png 

Processing Cell Type: EC 
  -> Loading genes...
      6mo : 265 genes
      PremTerm : 1149 genes
      Rej : 188 genes
      PCMV : 1116 genes
  -> Saved region summary: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/Venn/Venn_Data_Summary_